<a href="https://colab.research.google.com/github/jessie0707a/Machine-Learning-Project/blob/main/Predicting_Delivery_Duration_for_Olist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
#Mount the googledrive to access files
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


Merge tables

In [11]:
import pandas as pd
import os

# Base path for the datasets (assuming the CSVs are directly in this directory)
# Re-setting to the known working path based on successful previous executions.
base_path = "/content/drive/MyDrive/Machine learning/Olist order data/"

# Define file paths for each dataset
orders_path = os.path.join(base_path, "olist_orders_dataset.csv")
order_items_path = os.path.join(base_path, "olist_order_items_dataset.csv")
customers_path = os.path.join(base_path, "olist_customers_dataset.csv")
sellers_path = os.path.join(base_path, "olist_sellers_dataset.csv")
products_path = os.path.join(base_path, "olist_products_dataset.csv")
geolocation_path = os.path.join(base_path, "olist_geolocation_dataset.csv")

# Load datasets
try:
    orders_df = pd.read_csv(orders_path)
    order_items_df = pd.read_csv(order_items_path)
    customers_df = pd.read_csv(customers_path)
    sellers_df = pd.read_csv(sellers_path)
    products_df = pd.read_csv(products_path)
    geolocation_df = pd.read_csv(geolocation_path)
    print("All datasets loaded successfully.")

    # Pre-process geolocation data: aggregate by zip code to get mean lat/lng
    # This prevents an explosion of rows when merging, as each zip code can have multiple entries.
    geolocation_df_agg = geolocation_df.groupby('geolocation_zip_code_prefix').agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean')
    ).reset_index()

    # Start merging: orders and customers
    df_combined = pd.merge(orders_df, customers_df, on='customer_id', how='left')

    # Merge with order_items
    df_combined = pd.merge(df_combined, order_items_df, on='order_id', how='left')

    # Merge with products
    df_combined = pd.merge(df_combined, products_df, on='product_id', how='left')

    # Merge with sellers
    df_combined = pd.merge(df_combined, sellers_df, on='seller_id', how='left')

    # Merge with customer geolocation
    # Rename geolocation columns to distinguish from seller geolocation
    customer_geolocation_df = geolocation_df_agg.rename(columns={
        'geolocation_lat': 'customer_geolocation_lat',
        'geolocation_lng': 'customer_geolocation_lng'
    })
    df_combined = pd.merge(
        df_combined,
        customer_geolocation_df,
        left_on='customer_zip_code_prefix',
        right_on='geolocation_zip_code_prefix',
        how='left'
    )
    # Drop the redundant geolocation_zip_code_prefix column from the merged dataframe
    df_combined.drop('geolocation_zip_code_prefix', axis=1, inplace=True)

    # Merge with seller geolocation
    # Rename geolocation columns for sellers
    seller_geolocation_df = geolocation_df_agg.rename(columns={
        'geolocation_lat': 'seller_geolocation_lat',
        'geolocation_lng': 'seller_geolocation_lng'
    })
    df_combined = pd.merge(
        df_combined,
        seller_geolocation_df,
        left_on='seller_zip_code_prefix',
        right_on='geolocation_zip_code_prefix',
        how='left'
    )
    # Drop the redundant geolocation_zip_code_prefix column from the merged dataframe
    df_combined.drop('geolocation_zip_code_prefix', axis=1, inplace=True)

    print(f"Combined DataFrame created with {df_combined.shape[0]} rows and {df_combined.shape[1]} columns.")
    print("First 5 rows of the combined DataFrame:")
    print(df_combined.head())

except FileNotFoundError as e:
    print(f"Error: One of the specified files was not found. Please check the path and filename: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

All datasets loaded successfully.
Combined DataFrame created with 113425 rows and 33 columns.
First 5 rows of the combined DataFrame:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_deliv

In [12]:
display(df_combined.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,customer_geolocation_lat,customer_geolocation_lng,seller_geolocation_lat,seller_geolocation_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,19.0,8.0,13.0,9350.0,maua,SP,-23.576983,-46.587161,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,19.0,13.0,19.0,31570.0,belo horizonte,SP,-12.177924,-44.660711,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,24.0,19.0,21.0,14840.0,guariba,SP,-16.745150,-48.514783,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,30.0,10.0,20.0,31842.0,belo horizonte,MG,-5.774190,-35.271143,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,51.0,15.0,15.0,8752.0,mogi das cruzes,SP,-23.676370,-46.514627,-23.543395,-46.262086


In [13]:
print(df_combined.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_geolocation_lat', 'customer_geolocation_lng', 'seller_geolocation_lat', 'seller_geolocation_lng']


filter the `df_combined` to keep only the columns specified

In [32]:
columns_to_keep = [
    'order_id',
    'customer_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date', # Kept for target variable calculation
    'order_estimated_delivery_date',
    'product_id',
    'seller_id',
    'shipping_limit_date',
    'price',
    'freight_value',
    'customer_zip_code_prefix',
    'customer_state',
    'seller_zip_code_prefix',
    'seller_state',
    'customer_geolocation_lat',
    'customer_geolocation_lng',
    'seller_geolocation_lat',
    'seller_geolocation_lng',
    'product_category_name',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
    # 'payment_type', # Removed as per user request
    # 'payment_installments' # Removed as per user request
]

# Filter df_combined to keep only the specified columns
df_processed = df_combined[columns_to_keep].copy()

# Convert date columns to datetime objects for easier manipulation
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'shipping_limit_date']
for col in date_cols:
    if col in df_processed.columns:
        df_processed[col] = pd.to_datetime(df_processed[col], errors='coerce')

print(f"Processed DataFrame created with {df_processed.shape[0]} rows and {df_processed.shape[1]} columns.")
print("First 5 rows of the processed DataFrame:")
display(df_processed.head())

print("Column data types of the processed DataFrame:")
display(df_processed.info())

Processed DataFrame created with 113425 rows and 23 columns.
First 5 rows of the processed DataFrame:


,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,product_id,seller_id,shipping_limit_date,price,freight_value,...,seller_state,customer_geolocation_lat,customer_geolocation_lng,seller_geolocation_lat,seller_geolocation_lng,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,...,SP,-23.576983,-46.587161,-23.680729,-46.444238,utilidades_domesticas,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,...,SP,-12.177924,-44.660711,-19.807681,-43.980427,perfumaria,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,...,SP,-16.745150,-48.514783,-21.363502,-48.229601,automotivo,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,...,MG,-5.774190,-35.271143,-19.837682,-43.924053,pet_shop,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,...,SP,-23.676370,-46.514627,-23.543395,-46.262086,papelaria,250.0,51.0,15.0,15.0


Column data types of the processed DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  object        
 1   customer_id                    113425 non-null  object        
 2   order_purchase_timestamp       113425 non-null  datetime64[ns]
 3   order_delivered_customer_date  110196 non-null  datetime64[ns]
 4   order_estimated_delivery_date  113425 non-null  datetime64[ns]
 5   product_id                     112650 non-null  object        
 6   seller_id                      112650 non-null  object        
 7   shipping_limit_date            112650 non-null  datetime64[ns]
 8   price                          112650 non-null  float64       
 9   freight_value                  112650 non-null  float64       
 10  customer_zip_code_pref

None

In [15]:
output_path = os.path.join(base_path, "df_processed.csv")
df_processed.to_csv(output_path, index=False)
print(f"df_processed saved to {output_path}")

df_processed saved to /content/drive/MyDrive/Machine learning/Olist order data/df_processed.csv


In [16]:
# Filter out rows where 'order_delivered_customer_date' is null
df_processed = df_processed.dropna(subset=['order_delivered_customer_date']).copy()

print(f"Filtered DataFrame has {df_processed.shape[0]} rows and {df_processed.shape[1]} columns after removing null delivery dates.")
print("First 5 rows of the filtered DataFrame:")
display(df_processed.head())

print("Column data types of the filtered DataFrame:")
display(df_processed.info())

Filtered DataFrame has 110196 rows and 23 columns after removing null delivery dates.
First 5 rows of the filtered DataFrame:


,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,product_id,seller_id,shipping_limit_date,price,freight_value,...,seller_state,customer_geolocation_lat,customer_geolocation_lng,seller_geolocation_lat,seller_geolocation_lng,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,...,SP,-23.576983,-46.587161,-23.680729,-46.444238,utilidades_domesticas,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,...,SP,-12.177924,-44.660711,-19.807681,-43.980427,perfumaria,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,...,SP,-16.745150,-48.514783,-21.363502,-48.229601,automotivo,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,...,MG,-5.774190,-35.271143,-19.837682,-43.924053,pet_shop,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,...,SP,-23.676370,-46.514627,-23.543395,-46.262086,papelaria,250.0,51.0,15.0,15.0


Column data types of the filtered DataFrame:
<class 'pandas.core.frame.DataFrame'>
Index: 110196 entries, 0 to 113424
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       110196 non-null  object        
 1   customer_id                    110196 non-null  object        
 2   order_purchase_timestamp       110196 non-null  datetime64[ns]
 3   order_delivered_customer_date  110196 non-null  datetime64[ns]
 4   order_estimated_delivery_date  110196 non-null  datetime64[ns]
 5   product_id                     110196 non-null  object        
 6   seller_id                      110196 non-null  object        
 7   shipping_limit_date            110196 non-null  datetime64[ns]
 8   price                          110196 non-null  float64       
 9   freight_value                  110196 non-null  float64       
 10  customer_zip_code_prefix    

None

Identify what share of orders contain items from more than one product category

Result:
"This project scopes the modelling task to single-category orders (94,420 of 96,476 orders, 97.87%), excluding multi-category orders (0.75%) and orders with missing category data (1.38%) to keep the product-category feature unambiguous. The trained model is therefore validated only on single-category orders and its performance on multi-category orders is untested." *italicized text*

In [25]:
# Check what share of orders contain items from more than one product category
category_diversity = (
    df_processed.groupby('order_id')['product_category_name']
    .nunique()
    .rename('num_unique_categories')
)

total_orders = len(category_diversity)
multi_category_orders = (category_diversity > 1).sum()
single_category_orders = (category_diversity == 1).sum()

print(f"Total orders: {total_orders}")
print(f"Single-category orders: {single_category_orders} ({single_category_orders / total_orders:.2%})")
print(f"Multi-category orders: {multi_category_orders} ({multi_category_orders / total_orders:.2%})")

# See the full distribution (1 category, 2 categories, 3 categories, ...)
print("\nDistribution of number of distinct categories per order:")
print(category_diversity.value_counts().sort_index())

Total orders: 96476
Single-category orders: 94420 (97.87%)
Multi-category orders: 724 (0.75%)

Distribution of number of distinct categories per order:
num_unique_categories
0     1332
1    94420
2      709
3       15
Name: count, dtype: int64


### Step 1: Filter `df_processed` to retain single-product category orders

In [33]:
# Calculate the number of unique product categories for each order
category_diversity_processed = (
    df_processed.groupby('order_id')['product_category_name']
    .nunique()
    .rename('num_unique_categories')
)

# Find order_ids that contain only one product category
single_category_order_ids = category_diversity_processed[category_diversity_processed == 1].index

# Filter df_processed
df_single_category_orders = df_processed[df_processed['order_id'].isin(single_category_order_ids)].copy()

print(f"Number of unique orders in df_processed before filtering: {df_processed['order_id'].nunique()}")
print(f"Number of unique orders in df_single_category_orders after filtering: {df_single_category_orders['order_id'].nunique()}")
print(f"Shape of df_single_category_orders: {df_single_category_orders.shape}")
display(df_single_category_orders.head())

Number of unique orders in df_processed before filtering: 99441
Number of unique orders in df_single_category_orders after filtering: 96550
Shape of df_single_category_orders: (109357, 23)


,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,product_id,seller_id,shipping_limit_date,price,freight_value,...,seller_state,customer_geolocation_lat,customer_geolocation_lng,seller_geolocation_lat,seller_geolocation_lng,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,...,SP,-23.576983,-46.587161,-23.680729,-46.444238,utilidades_domesticas,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,...,SP,-12.177924,-44.660711,-19.807681,-43.980427,perfumaria,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,...,SP,-16.745150,-48.514783,-21.363502,-48.229601,automotivo,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,...,MG,-5.774190,-35.271143,-19.837682,-43.924053,pet_shop,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,...,SP,-23.676370,-46.514627,-23.543395,-46.262086,papelaria,250.0,51.0,15.0,15.0


### Step 2: Calculate Haversine distance between customer and seller

In [34]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius (km)

    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    distance = R * c
    return distance

# Calculate distance and handle missing values (e.g., if geolocation info is missing, distance will also be NaN)
# This will operate on df_single_category_orders
df_single_category_orders['distance_km'] = haversine_distance(
    df_single_category_orders['customer_geolocation_lat'],
    df_single_category_orders['customer_geolocation_lng'],
    df_single_category_orders['seller_geolocation_lat'],
    df_single_category_orders['seller_geolocation_lng']
)

print("'distance_km' column has been calculated.")
display(df_single_category_orders[['order_id', 'customer_geolocation_lat', 'customer_geolocation_lng', 'seller_geolocation_lat', 'seller_geolocation_lng', 'distance_km']].head())

'distance_km' column has been calculated.


,order_id,customer_geolocation_lat,customer_geolocation_lng,seller_geolocation_lat,seller_geolocation_lng,distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,-23.576983,-46.587161,-23.680729,-46.444238,18.576110
1,53cdb2fc8bc7dce0b6741e2150273451,-12.177924,-44.660711,-19.807681,-43.980427,851.495069
2,47770eb9100c2d0c44946d9cf07ec65d,-16.745150,-48.514783,-21.363502,-48.229601,514.410666
3,949d5b44dbf5de918fe9c16f97b45f8a,-5.774190,-35.271143,-19.837682,-43.924053,1822.226336
4,ad21c59c0840e6cb83a9ceb5573f8159,-23.676370,-46.514627,-23.543395,-46.262086,29.676625


### Step 3: Recreate `df_order_level` using the updated aggregation dictionary

In [35]:
# Redefine aggregation dictionary
agg_dict_new = {
    'customer_id': 'first', # Identifier
    'order_purchase_timestamp': 'first', # Datetime, unique per order
    'order_delivered_customer_date': 'first', # Datetime, unique per order
    'order_estimated_delivery_date': 'first', # Datetime, unique per order

    'product_id': 'nunique', # Count unique products per order
    'seller_id': 'nunique', # Count unique sellers per order
    'shipping_limit_date': 'max', # Latest shipping deadline for the order

    'price': 'sum', # Sum total price for all items in the order
    'freight_value': 'sum', # Sum total freight for all items in the order

    'customer_zip_code_prefix': 'first', # Numerical but acts as identifier, consistent per order
    'customer_state': 'first', # Categorical, consistent per order

    'customer_geolocation_lat': 'first', # Consistent per order after zip prefix aggregation
    'customer_geolocation_lng': 'first', # Consistent per order after zip prefix aggregation

    'distance_km': 'max', # Take the maximum distance, as an order is considered 'delivered' only when all seller items are delivered; thus, the furthest seller might be the bottleneck.

    'product_category_name': 'first', # Each order now contains only one category, so taking the first is sufficient
    'product_weight_g': 'sum', # Sum total weight of all products in the order
    'product_length_cm': 'max', # Max length for packaging
    'product_height_cm': 'max', # Max height for packaging
    'product_width_cm': 'max' # Max width for packaging
}

# Aggregate using the new df_single_category_orders and updated agg_dict
df_order_level = df_single_category_orders.groupby('order_id').agg(agg_dict_new).reset_index()

# Rename columns for clarity
df_order_level.rename(columns={
    'product_id': 'num_unique_products',
    'seller_id': 'num_unique_sellers',
    'price': 'total_price',
    'freight_value': 'total_freight_value',
    'product_weight_g': 'total_product_weight_g',
    'product_length_cm': 'max_product_length_cm',
    'product_height_cm': 'max_product_height_cm',
    'product_width_cm': 'max_product_width_cm'
}, inplace=True)

print(f"Number of unique orders in the original df_processed: {df_processed['order_id'].nunique()}")
print(f"Number of orders after filtering for single-category orders: {df_single_category_orders['order_id'].nunique()}")
print(f"Aggregated df_order_level has {df_order_level.shape[0]} rows and {df_order_level.shape[1]} columns.")
print("First 5 rows of the aggregated DataFrame:")
display(df_order_level.head())

Number of unique orders in the original df_processed: 99441
Number of orders after filtering for single-category orders: 96550
Aggregated df_order_level has 96550 rows and 20 columns.
First 5 rows of the aggregated DataFrame:


,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,num_unique_products,num_unique_sellers,shipping_limit_date,total_price,total_freight_value,customer_zip_code_prefix,customer_state,customer_geolocation_lat,customer_geolocation_lng,distance_km,product_category_name,total_product_weight_g,max_product_length_cm,max_product_height_cm,max_product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,1,1,2017-09-19 09:45:35,58.90,13.29,28013,RJ,-21.762775,-41.309633,301.504681,cool_stuff,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,1,1,2017-05-03 11:05:13,239.90,19.93,15775,SP,-20.220527,-50.903424,585.563937,pet_shop,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,1,1,2018-01-18 14:48:30,199.00,17.87,35661,MG,-19.870305,-44.593326,312.343511,moveis_decoracao,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,1,1,2018-08-15 10:10:18,12.99,12.79,12952,SP,-23.089925,-46.611654,293.168420,perfumaria,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,1,1,2017-02-13 13:57:51,199.90,18.14,13226,SP,-23.243402,-46.827614,646.163463,ferramentas_jardim,3750.0,35.0,40.0,30.0


### Check for missing values in the re-aggregated `df_order_level`

In [36]:
print("Number and percentage of missing values per column in the re-aggregated df_order_level:")
missing_values_new = df_order_level.isnull().sum()
missing_percentage_new = (df_order_level.isnull().sum() / len(df_order_level)) * 100

missing_info_new = pd.DataFrame({
    'Missing Values': missing_values_new,
    'Percentage': missing_percentage_new
})

display(missing_info_new[missing_info_new['Missing Values'] > 0].sort_values(by='Missing Values', ascending=False))

Number and percentage of missing values per column in the re-aggregated df_order_level:


,Missing Values,Percentage
order_delivered_customer_date,2130,2.206111
distance_km,480,0.497152
customer_geolocation_lat,266,0.275505
customer_geolocation_lng,266,0.275505
max_product_length_cm,1,0.001036
max_product_height_cm,1,0.001036
max_product_width_cm,1,0.001036


### Handle remaining missing values in the re-aggregated `df_order_level`

In [37]:
# Handle missing values in df_order_level

# 1. Handle missing values in 'product_category_name': impute with mode
# mode()[0] is used to handle cases where there might be multiple modes; we take the first one.
if 'product_category_name' in df_order_level.columns:
    # Check if there are missing values to impute; if all values are NaN, mode() might return an empty Series
    if not df_order_level['product_category_name'].mode().empty:
        mode_product_category = df_order_level['product_category_name'].mode()[0]
        df_order_level['product_category_name'].fillna(mode_product_category, inplace=True)
        print(f"Missing values in 'product_category_name' have been imputed with mode '{mode_product_category}'.")
    else:
        print("'product_category_name' has no valid mode for imputation, or all values are NaN.")

# 2. Handle missing values in numerical geolocation, distance, and product dimension columns: impute with median
# Identify numerical columns that need median imputation
numerical_cols_to_impute_new = [
    'distance_km',
    'customer_geolocation_lat',
    'customer_geolocation_lng',
    'total_product_weight_g',
    'max_product_length_cm',
    'max_product_height_cm',
    'max_product_width_cm'
]

for col in numerical_cols_to_impute_new:
    if col in df_order_level.columns:
        if df_order_level[col].isnull().any(): # Only calculate median and impute if missing values exist
            median_value = df_order_level[col].median()
            df_order_level[col].fillna(median_value, inplace=True)
            print(f"Missing values in '{col}' have been imputed with median {median_value}.")
        else:
            print(f"'{col}' has no missing values to impute.")

print("\nMissing values status of df_order_level after imputation:")
missing_values_after_imputation_new = df_order_level.isnull().sum()
missing_percentage_after_imputation_new = (df_order_level.isnull().sum() / len(df_order_level)) * 100

missing_info_after_imputation_new = pd.DataFrame({
    'Missing Values': missing_values_after_imputation_new,
    'Percentage': missing_percentage_after_imputation_new
})

display(missing_info_after_imputation_new[missing_info_after_imputation_new['Missing Values'] > 0].sort_values(by='Missing Values', ascending=False))

Missing values in 'product_category_name' have been imputed with mode 'cama_mesa_banho'.
Missing values in 'distance_km' have been imputed with median 435.3211978105313.
Missing values in 'customer_geolocation_lat' have been imputed with median -22.92401542829238.
Missing values in 'customer_geolocation_lng' have been imputed with median -46.63008918820539.
'total_product_weight_g' has no missing values to impute.
Missing values in 'max_product_length_cm' have been imputed with median 25.0.
Missing values in 'max_product_height_cm' have been imputed with median 13.0.
Missing values in 'max_product_width_cm' have been imputed with median 20.0.

Missing values status of df_order_level after imputation:


/tmp/ipykernel_1228/1634743742.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_order_level['product_category_name'].fillna(mode_product_category, inplace=True)
/tmp/ipykernel_1228/1634743742.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col

,Missing Values,Percentage
order_delivered_customer_date,2130,2.206111


### Check for missing values in the re-aggregated `df_order_level`

In [38]:
print("Number and percentage of missing values per column in the re-aggregated df_order_level:")
missing_values_new = df_order_level.isnull().sum()
missing_percentage_new = (df_order_level.isnull().sum() / len(df_order_level)) * 100

missing_info_new = pd.DataFrame({
    'Missing Values': missing_values_new,
    'Percentage': missing_percentage_new
})

display(missing_info_new[missing_info_new['Missing Values'] > 0].sort_values(by='Missing Values', ascending=False))

Number and percentage of missing values per column in the re-aggregated df_order_level:


,Missing Values,Percentage
order_delivered_customer_date,2130,2.206111


In [19]:
df_processed_100_rows = df_processed.head(100).copy()
output_path_100_rows = os.path.join(base_path, "df_processed_100_rows.csv")
df_processed_100_rows.to_csv(output_path_100_rows, index=False)
print(f"First 100 rows of df_processed saved to {output_path_100_rows}")

First 100 rows of df_processed saved to /content/drive/MyDrive/Machine learning/Olist order data/df_processed_100_rows.csv
